# Phase 1 — ALBERT-style Transformer language model

Fourth architecture for Phase 1 (lightweight / parameter-shared Transformer). Everything upstream of the model is reused, so results stay comparable with the decoder-only, encoder-decoder and fusion runs:

- **Tokenizer:** `legal_bpe_v3.pkl` (byte-level BPE, 40k vocab)
- **Data:** `train_v3.bin` / `val_v3.bin` (uint16 token ids, four sources mixed 25% each)
- **Task:** next-token prediction, `model(idx) -> logits`, validated on the same 40 windows as the other runs

Cells 1–5 are copied unchanged from the fusion notebook (outputs cleared).


In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("CS781_WRITE")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

In [ ]:
import re
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

from datasets import load_dataset


class LegalBPETokenizer:

    def __init__(self, model_path):

        with open(model_path, "rb") as f:
            model = pickle.load(f)

        self.vocab_size = model["vocab_size"]
        self.merges = model["merges"]                     # [((a, b), new_id), ...]
        self.lowercase = model.get("lowercase", False)
        self.special_tokens = model.get("special_tokens", [])
        self.pattern = re.compile(model["pattern"])

        # pair -> rank (lower rank = learned earlier = merged first)
        self.merge_rank = {
            tuple(pair): rank
            for rank, (pair, _) in enumerate(self.merges)
        }

        # id -> bytes table (O(1) decode)
        self.token_bytes = {i: bytes([i]) for i in range(256)}
        for (l, r), new_id in self.merges:
            self.token_bytes[new_id] = self.token_bytes[l] + self.token_bytes[r]

        base = 256 + len(self.merges)
        self.special_to_id = {t: base + i for i, t in enumerate(self.special_tokens)}
        self.id_to_special = {v: k for k, v in self.special_to_id.items()}
        self.eos_id = self.special_to_id.get("<|endoftext|>")

        self._special_re = (
            re.compile("(" + "|".join(re.escape(t) for t in self.special_tokens) + ")")
            if self.special_tokens else None
        )
        self._cache = {}

    def _encode_piece(self, piece_bytes):

        cached = self._cache.get(piece_bytes)
        if cached is not None:
            return cached

        ids = list(piece_bytes)

        while len(ids) > 1:
            best, best_rank = None, None
            for p in zip(ids, ids[1:]):
                r = self.merge_rank.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = p, r
            if best is None:
                break

            new_id = 256 + best_rank
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == best[0] and ids[i + 1] == best[1]:
                    out.append(new_id)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out

        if len(self._cache) < 500_000:
            self._cache[piece_bytes] = ids
        return ids

    def _encode_ordinary(self, text):

        if self.lowercase:
            text = text.lower()

        out = []
        for piece in self.pattern.findall(text):
            out.extend(self._encode_piece(piece.encode("utf-8")))
        return out

    def encode(self, text, add_eos=False):

        if self._special_re is not None:
            ids = []
            for part in self._special_re.split(text):
                if part in self.special_to_id:
                    ids.append(self.special_to_id[part])
                elif part:
                    ids.extend(self._encode_ordinary(part))
        else:
            ids = self._encode_ordinary(text)

        if add_eos:
            ids.append(self.eos_id)

        return ids

    def decode(self, ids):

        chunks = []
        for token_id in ids:
            if token_id in self.id_to_special:
                chunks.append(self.id_to_special[token_id].encode("utf-8"))
            else:
                chunks.append(self.token_bytes[token_id])

        return b"".join(chunks).decode("utf-8", errors="replace")


TOKENIZER_PATH = "/kaggle/input/models/harsvij/cap-tok/pytorch/default/1/legal_bpe_v3.pkl"   # <-- change this

tokenizer = LegalBPETokenizer(TOKENIZER_PATH)

print("Vocabulary size:", tokenizer.vocab_size)
print([tokenizer.decode([i]) for i in tokenizer.encode("dog ate cat")])

In [ ]:
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

supreme_court_dataset = load_dataset(
    "json",
    data_files={
        "train": f"hf://datasets/{HF_DATASET_REPO}/Phase_1_Pretraining/Train/train_supreme_court.json"
    },
    split="train",
    streaming=True
)

print("SC dataset loaded.")

In [ ]:
%%writefile model_arch_albert.py

"""ALBERT-style causal language model (skeleton).

Spec: 12 logical layers = 4 parameter-sharing groups x 3 layers per group,
d_model 1024, 16 heads, d_ff 4096, embedding size 128, learned absolute
positions, Pre-LayerNorm, GELU, dropout 0.1, tied input/output embeddings,
context 1024, vocab 40,000  (~57M parameters).

TODO: implement the modules below yourself.
"""

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class AlbertConfig:

    vocab_size: int = 40_000

    n_layers: int = 12            # logical (compute) depth
    n_groups: int = 4             # unique parameter sets
    layers_per_group: int = 3     # n_groups * layers_per_group must equal n_layers

    d_model: int = 1024
    n_heads: int = 16
    d_ff: int = 4096
    embed_size: int = 128         # factorised embedding size

    max_seq_len: int = 1024
    dropout: float = 0.1
    norm_eps: float = 1e-5

    def __post_init__(self):
        assert self.n_groups * self.layers_per_group == self.n_layers
        assert self.d_model % self.n_heads == 0


class CausalSelfAttention(nn.Module):
    """Causal multi-head self-attention. TODO"""

    def __init__(self, cfg: AlbertConfig):
        super().__init__()
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError


class FeedForward(nn.Module):
    """d_model -> d_ff -> d_model with GELU. TODO"""

    def __init__(self, cfg: AlbertConfig):
        super().__init__()
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError


class AlbertBlock(nn.Module):
    """One Pre-LayerNorm Transformer block: x + Attn(LN(x)); x + FFN(LN(x)). TODO"""

    def __init__(self, cfg: AlbertConfig):
        super().__init__()
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError


class AlbertLM(nn.Module):
    """Contract used by the rest of the notebook:

    forward(idx: LongTensor (B, T)) -> logits FloatTensor (B, T, vocab_size)
      * strictly causal: logits[:, t] depends only on idx[:, :t+1]
      * works for any 1 <= T <= cfg.max_seq_len
      * exposes self.cfg (needs .max_seq_len)

    TODO: factorised token embedding (V x embed_size) -> projection to d_model,
    learned absolute positions, `n_groups` UNIQUE AlbertBlocks each applied
    `layers_per_group` times, final LayerNorm, projection back to embed_size,
    output head tied to the token embedding, weight init.
    """

    def __init__(self, cfg: AlbertConfig):
        super().__init__()
        self.cfg = cfg
        raise NotImplementedError

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        assert 1 <= T <= self.cfg.max_seq_len, f"sequence length {T} outside [1, {self.cfg.max_seq_len}]"
        raise NotImplementedError


In [ ]:
import importlib, sys

sys.modules.pop("model_arch_albert", None)

model_arch_albert = importlib.import_module("model_arch_albert")

AlbertConfig = model_arch_albert.AlbertConfig
AlbertLM = model_arch_albert.AlbertLM

print("ALBERT model code loaded.")


## Data — same `train_v3.bin` / `val_v3.bin` as the earlier models

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset

TRAIN_PATH = "/kaggle/input/models/harsvij/cap-tok/pytorch/default/1/train_v3.bin"
VAL_PATH = "/kaggle/input/models/harsvij/cap-tok/pytorch/default/1/val_v3.bin"

SEQ_LEN = 1024          # same context as the other runs
BATCH_SIZE = 2          # micro-batch; gradient accumulation is set in the training cell
N_VAL_WINDOWS = 40      # with SEQ_LEN = 1024 these are the same windows the decoder-only run used


class TokenDataset(Dataset):

    def __init__(self, path, seq_len):
        self.tokens = np.memmap(path, dtype=np.uint16, mode="r")
        self.seq_len = seq_len

    def __len__(self):
        return len(self.tokens) - self.seq_len

    def __getitem__(self, idx):
        chunk = torch.from_numpy(
            np.asarray(self.tokens[idx:idx + self.seq_len + 1], dtype=np.int64)
        )
        return chunk[:-1], chunk[1:]          # inputs, next-token targets


train_dataset = TokenDataset(TRAIN_PATH, SEQ_LEN)
val_dataset = TokenDataset(VAL_PATH, SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)

val_idx = np.random.default_rng(0).integers(0, len(val_dataset), size=N_VAL_WINDOWS).tolist()
val_loader = DataLoader(Subset(val_dataset, val_idx), batch_size=BATCH_SIZE, shuffle=False)

print("Train tokens:", len(train_dataset))
print("Validation tokens:", len(val_dataset))

In [ ]:
VOCAB_SIZE = tokenizer.vocab_size

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = AlbertConfig(
    vocab_size=VOCAB_SIZE,
    max_seq_len=SEQ_LEN,
    n_layers=12,
    n_groups=4,
    layers_per_group=3,
    d_model=1024,
    n_heads=16,
    d_ff=4096,
    embed_size=128,
    dropout=0.1,
)

model = AlbertLM(cfg).to(device)

n_params = sum(p.numel() for p in model.parameters())      # tied weights are counted once

print(f"Parameters: {n_params:,} ({n_params / 1e6:.1f}M)   [target: about 57M; decoder-only / enc-dec baselines: about 111M]")
print("Device:", device)

assert 50e6 <= n_params <= 300e6, "Phase-1 rule: the model must have between 50M and 300M parameters"

### Sanity checks — run these before any long training run
Output shape, **causality** (a leak here gives a deceptively low loss and garbage generations), variable sequence length, and a one-batch overfit test.

In [ ]:
import copy
import torch.nn.functional as F

V = tokenizer.vocab_size

# 1) output shape
model.eval()
x = torch.randint(0, V, (2, 64), device=device)
with torch.no_grad():
    out = model(x)
assert out.shape == (2, 64, V), f"expected (2, 64, {V}), got {tuple(out.shape)}"

# 2) causality: changing the tokens AFTER position t must not change the logits up to t
t = 31
x2 = x.clone()
x2[:, t + 1:] = torch.randint(0, V, (2, 64 - t - 1), device=device)
with torch.no_grad():
    out2 = model(x2)
leak = (out[:, :t + 1] - out2[:, :t + 1]).abs().max().item()
moved = (out[:, t + 1:] - out2[:, t + 1:]).abs().max().item()
print(f"change on the unchanged prefix (must be ~0): {leak:.2e} | change after t (must be > 0): {moved:.2e}")
assert leak < 1e-4, "information leaks from future tokens: the model is not causal"
assert moved > 0, "the later positions ignore their input"

# 3) variable length (generation feeds growing sequences; the first call can be very short)
for T in (1, 17, SEQ_LEN):
    with torch.no_grad():
        assert model(torch.randint(0, V, (1, T), device=device)).shape == (1, T, V)

# 4) one-batch overfit: a healthy model memorises a single batch (loss falls far below ln(V) ~ 10.6)
RUN_OVERFIT_TEST = True
if RUN_OVERFIT_TEST:
    tmp = copy.deepcopy(model).train()
    opt = torch.optim.AdamW(tmp.parameters(), lr=3e-4)
    xb, yb = next(iter(train_loader))
    xb, yb = xb.to(device), yb.to(device)
    for i in range(1, 101):
        opt.zero_grad(set_to_none=True)
        loss = F.cross_entropy(tmp(xb).float().reshape(-1, V), yb.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(tmp.parameters(), 1.0)
        opt.step()
        if i in (1, 25, 50, 75, 100):
            print(f"overfit step {i:>3} | loss {loss.item():.3f}")
    del tmp, opt
    torch.cuda.empty_cache()

model.train()
print("sanity checks passed")

## Training
Same recipe as the earlier runs: AdamW (0.9, 0.95), linear warm-up + cosine decay to 10% of peak, fp16 autocast, gradient accumulation, gradient clipping, resumable checkpoints. The values below are **placeholders carried over from the fusion notebook** — choose and tune them yourself (note this model uses dropout 0.1, shares weights across 12 passes, and has ~57M params, so the best LR / token budget will probably differ from the 111M runs).


In [ ]:
import time, os, math, json
import torch
import torch.nn.functional as F

V = tokenizer.vocab_size

# ---- hyper-parameters (placeholders: set / tune these yourself) ----
ACCUM_STEPS = 4                      # 2 x 4 x 1024 = 8,192 tokens per optimiser step (same as the enc-dec run)
LEARNING_RATE = 6e-4                 # placeholder (fusion run used 6e-4): tune yourself
WEIGHT_DECAY = 0.1
BETAS = (0.9, 0.95)
GRAD_CLIP = 1.0
USE_AMP = True                       # fp16 speed-up; set False if the loss becomes NaN
TARGET_TOKENS = 80_000_000          # about one pass over train_v3.bin
WARMUP_STEPS = 200
MIN_LR_RATIO = 0.1
EVAL_EVERY = 100
SAVE_EVERY = 3000
CKPT_DIR = "/kaggle/working/checkpoints_albert"
RESUME_FROM = None                   # after a crash: f"{CKPT_DIR}/last.pt"

TRAIN_STEPS = TARGET_TOKENS // (BATCH_SIZE * ACCUM_STEPS * SEQ_LEN)
LOG_PATH = f"{CKPT_DIR}/train_log_albert.json"
os.makedirs(CKPT_DIR, exist_ok=True)

decay    = [p for n, p in model.named_parameters() if p.requires_grad and p.dim() >= 2]
no_decay = [p for n, p in model.named_parameters() if p.requires_grad and p.dim() < 2]
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": WEIGHT_DECAY},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=LEARNING_RATE, betas=BETAS,
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
train_iter = iter(train_loader)


def lr_at(step):
    if step <= WARMUP_STEPS:
        return LEARNING_RATE * step / WARMUP_STEPS
    p = (step - WARMUP_STEPS) / max(1, TRAIN_STEPS - WARMUP_STEPS)
    return LEARNING_RATE * (MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * p)))


def next_batch(it, loader):
    try:
        return next(it), it
    except StopIteration:
        it = iter(loader)
        return next(it), it


@torch.no_grad()
def evaluate(m):
    m.eval()
    losses = []
    for xv, yv in val_loader:
        xv, yv = xv.to(device), yv.to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            vl = m(xv)
        losses.append(F.cross_entropy(vl.float().reshape(-1, V), yv.reshape(-1)).item())
        del vl
    m.train()
    return sum(losses) / len(losses)


def save_ckpt(path, step):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({"model_state": model.state_dict(), "config": vars(cfg), "vocab_size": V, "step": step,
                "optimizer_state": optimizer.state_dict(), "scaler_state": scaler.state_dict()}, path)


start_step = 0
history = []
if RESUME_FROM:
    ckpt = torch.load(RESUME_FROM, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    start_step = ckpt["step"]
    if "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scaler.load_state_dict(ckpt["scaler_state"])
    if os.path.exists(LOG_PATH):
        history = json.load(open(LOG_PATH))
    print("resumed from step", start_step)

print(f"tokens/step: {BATCH_SIZE * ACCUM_STEPS * SEQ_LEN:,} | steps: {TRAIN_STEPS:,} | peak LR: {LEARNING_RATE:.2e}")

model.train()
t0 = time.time()
for step in range(start_step + 1, TRAIN_STEPS + 1):
    for g in optimizer.param_groups:
        g["lr"] = lr_at(step)

    optimizer.zero_grad(set_to_none=True)
    train_loss = 0.0
    for _ in range(ACCUM_STEPS):
        (x, y), train_iter = next_batch(train_iter, train_loader)
        x, y = x.to(device), y.to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
        loss = F.cross_entropy(logits.float().reshape(-1, V), y.reshape(-1)) / ACCUM_STEPS
        scaler.scale(loss).backward()
        train_loss += loss.item()
        del logits, loss

    if not math.isfinite(train_loss):
        raise RuntimeError(f"loss is {train_loss} at step {step}: lower the LR, set USE_AMP = False, or check the model")

    scaler.unscale_(optimizer)
    gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(optimizer)
    scaler.update()

    if step == start_step + 1:
        print(f"first loss: {train_loss:.3f} (about ln(V) = {math.log(V):.1f} expected) | "
              f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

    if step % EVAL_EVERY == 0:
        val_loss = evaluate(model)
        history.append({"step": step, "train_loss": train_loss, "val_loss": val_loss,
                        "val_ppl": math.exp(val_loss), "lr": lr_at(step),
                        "grad_norm": float(gnorm), "minutes": (time.time() - t0) / 60})
        print(f"Step {step}/{TRAIN_STEPS} | Train {train_loss:.4f} | Val {val_loss:.4f} (ppl {math.exp(val_loss):.1f}) | "
              f"gnorm {float(gnorm):.2f} | LR {lr_at(step):.2e} | {(time.time() - t0) / 60:.1f} min")

    if step % SAVE_EVERY == 0:
        save_ckpt(f"{CKPT_DIR}/last.pt", step)
        json.dump(history, open(LOG_PATH, "w"))

save_ckpt(f"{CKPT_DIR}/albert_final.pt", TRAIN_STEPS)
json.dump(history, open(LOG_PATH, "w"))
print("done | final checkpoint:", f"{CKPT_DIR}/albert_final.pt")

In [ ]:
import matplotlib.pyplot as plt

steps = [h["step"] for h in history]

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))

ax[0].plot(steps, [h["train_loss"] for h in history], label="train")
ax[0].plot(steps, [h["val_loss"] for h in history], label="val")
ax[0].set(title="loss", xlabel="step")
ax[0].legend()

ax[1].plot(steps, [h["val_ppl"] for h in history])
ax[1].set(title="validation perplexity", xlabel="step", yscale="log")

ax[2].plot(steps, [h["grad_norm"] for h in history])
ax[2].set(title="gradient norm (before clipping)", xlabel="step")

plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import FileLink, display

display(FileLink(
    "checkpoints_albert/albert_final.pt",
    result_html_prefix="Download checkpoint: "
))

## Generation and submission
Same decoding as the earlier runs (temperature + top-k + repetition penalty, EOS blocked so every continuation reaches 128 words). Since `model(idx)` returns logits for the whole window, the generator works unchanged for any causal model that follows the contract above.

In [ ]:
import json
from huggingface_hub import hf_hub_download, HfApi

REPO = "Exploration-Lab/CS-781-Capstone"
files = [f for f in HfApi().list_repo_files(REPO, repo_type="dataset") if "Phase_1" in f and "Test" in f]
print(files)                                   # check this is the Phase 1 test file
test_data = json.load(open(hf_hub_download(REPO, files[0], repo_type="dataset"), encoding="utf-8"))
print(len(test_data), "prompts")

In [ ]:
block = cfg.max_seq_len        # 1024


@torch.no_grad()
def generate_batch(model, x, block, eos_id, temperature=0.8, top_k=50, rep_penalty=1.1, max_new_tokens=400):
    model.eval()
    B, T0 = x.shape
    for i in range(max_new_tokens):
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(x[:, -block:])
        logits = (out[0] if isinstance(out, tuple) else out)[:, -1, :].float()
        logits[:, eos_id] = -float("inf")                  # we need a full 128 words
        window = x[:, -256:]
        s = logits.gather(1, window)
        s = torch.where(s > 0, s / rep_penalty, s * rep_penalty)
        logits.scatter_(1, window, s)
        logits = logits / temperature
        kth = torch.topk(logits, top_k)[0][:, -1:]
        logits[logits < kth] = -float("inf")
        nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)
        x = torch.cat([x, nxt], dim=1)
        if (i + 1) % 16 == 0 and all(len(tokenizer.decode(r[T0:].tolist()).split()) > 128 for r in x):
            break
    return [tokenizer.decode(r[T0:].tolist()) for r in x]


def generate_texts(prompts, batch=8, prompt_tokens=800, **gen_kw):
    ids = [tokenizer.encode(p) for p in prompts]
    order = sorted(range(len(ids)), key=lambda i: len(ids[i]))
    outs = [None] * len(ids)
    for s in range(0, len(order), batch):
        idx = order[s:s + batch]
        L = min(prompt_tokens, block, min(len(ids[i]) for i in idx))
        x = torch.tensor([ids[i][-L:] for i in idx], device=device)
        for i, t in zip(idx, generate_batch(model, x, block, tokenizer.eos_id, **gen_kw)):
            outs[i] = " ".join(t.split()[:128])
    return outs

In [ ]:
from tqdm.auto import tqdm

SMOKE = True                 # True = 8 prompts only; set False for the full test set
PROMPT_TOKENS = 800
BATCH = 8
torch.manual_seed(42)

keys = list(test_data)[:8] if SMOKE else list(test_data)
ids = {k: tokenizer.encode(test_data[k]["text"]) for k in keys}
keys = sorted(keys, key=lambda k: len(ids[k]))          # similar lengths share a batch (no short prompt shrinks the others)

predictions = {}
for i in tqdm(range(0, len(keys), BATCH)):
    kb = keys[i:i + BATCH]
    L = min(PROMPT_TOKENS, block, min(len(ids[k]) for k in kb))
    x = torch.tensor([ids[k][-L:] for k in kb], device=device)
    for k, t in zip(kb, generate_batch(model, x, block, tokenizer.eos_id)):
        predictions[k] = {"text": " ".join(t.split()[:128])}

if SMOKE:
    for k in keys[:2]:
        print(predictions[k]["text"], "\n---")

In [ ]:
import zipfile

TAG = "albert_T0.8_k50"          # change per run so you keep separate zips
assert not SMOKE, "set SMOKE = False and re-run the generation cell first"

# 1) repair any prediction that came out shorter than 128 words
for attempt in range(5):
    bad = [k for k, v in predictions.items() if len(v["text"].split()) != 128]
    if not bad:
        break
    print(f"attempt {attempt + 1}: regenerating {len(bad)} prediction(s)")
    torch.manual_seed(1000 + attempt)
    for k in bad:
        L = min(PROMPT_TOKENS, block, len(ids[k]))
        x = torch.tensor([ids[k][-L:]], device=device)
        words = generate_batch(model, x, block, tokenizer.eos_id, max_new_tokens=700)[0].split()
        if len(words) >= 128:
            predictions[k] = {"text": " ".join(words[:128])}
        elif len(words) > len(predictions[k]["text"].split()):
            predictions[k] = {"text": " ".join(words)}

still_bad = [k for k, v in predictions.items() if len(v["text"].split()) != 128]
for k in still_bad:                                          # last resort: repeat its own words
    words = predictions[k]["text"].split()
    if words:
        predictions[k] = {"text": " ".join((words * (128 // len(words) + 1))[:128])}
print(f"{len(still_bad)} prediction(s) needed the repeat-words fallback")

# 2) validate, write, zip
assert set(predictions) == set(test_data)
bad = [k for k, v in predictions.items() if len(v["text"].split()) != 128]
assert not bad, f"{len(bad)} predictions are not exactly 128 words, e.g. {bad[:3]}"

with open("prediction.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False)
ZIP_NAME = f"submission_{TAG}.zip"
with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    z.write("prediction.json", arcname="prediction.json")
print(zipfile.ZipFile(ZIP_NAME).namelist())                  # must print ['prediction.json']